# Task 2 — QuadX-Waypoints-v4 with PPO

**Goal.** Train PPO to fly a quadrotor through a sequence of 4 randomly placed
3-D waypoints in PyFlyt's `QuadX-Waypoints-v4`, under **two flight modes**:

* **mode 0** — angular-rate + thrust commands (the agent must learn inner-loop
  stabilisation *and* outer-loop navigation simultaneously).
* **mode 6** — ground-frame velocity + yaw-rate + vertical-velocity commands
  (the cascaded PID handles inner-loop stabilisation; the agent only chooses where to go).

Comparing these two modes directly answers the project's *"why is mode 0 harder
than mode 6"* question.

**Pipeline.**

1. Configure environment (with `FlattenWaypointEnv` and the override kwargs from `env_config.py`).
2. Train PPO for $5 \times 2 = 10$ runs (5 seeds × 2 modes), $10^6$ env steps each.
3. Plot per-mode learning curves with bootstrap CIs, plus a side-by-side comparison.
4. Re-evaluate each final checkpoint over 20 deterministic episodes, recording mean return,
   crash rate, and **mean number of waypoints reached** out of 4.
5. Aggregate statistics (mean / IQM / bootstrap CI) per mode.

**Compute.** ~150 minutes total at $5 \times 2 \times \sim\!15$ minutes per seed on a modern CPU.
Each seed runs in its own `multiprocessing` subprocess to immunise the training
loop against PyBullet socket leaks (the same fix we applied in Task 1).

## 1. Imports and paths

In [ ]:
import sys

# Install missing dependencies for the QuadX-Hover-v4 task
!pip install PyFlyt stable-baselines3 gymnasium shimmy


In [ ]:
from __future__ import annotations

import json
import os
import random
import sys
from pathlib import Path

# Attempt to handle numpy/gymnasium version conflicts after pip install
# If numpy was downgraded in a previous cell, modules might still hold references
# to the old binary. Removing from sys.modules forces a fresh import.
# Removed explicit deletion of numpy and gymnasium from sys.modules to prevent circular import issues.
# if 'numpy' in sys.modules:
#     del sys.modules['numpy']
# if 'gymnasium' in sys.modules:
#     del sys.modules['gymnasium']

import gymnasium
import numpy as np
import torch

import PyFlyt.gym_envs  # noqa: F401  (registers PyFlyt envs as a side effect)

import stable_baselines3 as sb3
from stable_baselines3 import PPO     # for the post-training evaluation cells

# Make scripts/env_config.py and scripts/wrappers.py importable.
PROJECT_ROOT = Path.cwd().resolve()
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

# Required for Waypoints: env-config overrides + Dict-obs flattening wrapper
# (used by the post-training evaluation, not by the training subprocess).
from env_config import get_env_kwargs        # noqa: E402
from wrappers import FlattenWaypointEnv      # noqa: E402

# Note: training is delegated to training_cell_v2.py (imported in cell 4),
# so the SB3 callbacks/wrappers used during training (EvalCallback, Monitor,
# DummyVecEnv, etc.) are imported there, not here.

print(f"Stable-Baselines3 version: {sb3.__version__}")
print(f"PyTorch version:           {torch.__version__}")
print(f"CUDA available:            {torch.cuda.is_available()}")
print(f"Project root:              {PROJECT_ROOT}")
print(f"Waypoint env_kwargs:       {get_env_kwargs('waypoints')}")

In [ ]:
import os
# Redémarrage forcé du runtime pour nettoyer l'état des modules
os._exit(00)

## 2. Configuration

All hyperparameters live in one place. The two flight modes we compare
are listed in `MODES`; iterating over this list everywhere makes adding a third
mode (or removing one) a one-line change.

In [ ]:
# ---- Experiment identity ---------------------------------------------------
ENV_ID         = "PyFlyt/QuadX-Waypoints-v4"
ENV_SHORT      = "QuadX-Waypoints-v4"
ALGO_NAME      = "PPO"
MODES          = [0, 6]                 # the two flight modes we compare

# ---- Compute budget --------------------------------------------------------
SEEDS           = [0, 1, 2, 3, 4]
TOTAL_TIMESTEPS = 1_000_000             # 2x what we used for Hover
N_ENVS          = 4
EVAL_FREQ       = 25_000                # vec-env steps; halves PyBullet churn vs 10k
N_EVAL_EPISODES = 5                     # eval rollouts per evaluation point during training
FINAL_EVAL_EPISODES = 20                # post-training evaluation episodes per checkpoint

# ---- PPO hyperparameters ---------------------------------------------------
# Identical to Task 1, except n_steps doubled to 4096. Justification: Waypoints
# episodes are 3600 steps (120s @ 30Hz), versus 402 for Hover. With n_steps=2048
# per env and 4 envs, a single rollout would only see ~2 waypoint capture events
# on average. Bumping to 4096 lifts that to ~4 events per rollout, giving GAE a
# cleaner advantage signal without increasing wall-clock cost much (one update
# every 16k transitions instead of every 8k).
PPO_KWARGS = dict(
    policy="MlpPolicy",
    learning_rate=3e-4,
    n_steps=4096,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.0,
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs=dict(net_arch=dict(pi=[64, 64], vf=[64, 64])),
    verbose=0,
)

# ---- Environment overrides (from scripts/env_config.py) --------------------
WAYPOINT_KWARGS = get_env_kwargs("waypoints")
# {goal_reach_distance: 4.0, flight_dome_size: 150.0,
#  max_duration_seconds: 120.0, num_targets: 4}
MAX_WAYPOINTS = WAYPOINT_KWARGS["num_targets"]   # used by FlattenWaypointEnv

# ---- Output paths ----------------------------------------------------------
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR  = RESULTS_DIR / "models"
LOGS_DIR    = RESULTS_DIR / "logs"
EVAL_DIR    = RESULTS_DIR / "eval"
FIGURES_DIR = RESULTS_DIR / "figures"
for d in (MODELS_DIR, LOGS_DIR, EVAL_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---- Behaviour flags -------------------------------------------------------
FORCE_RETRAIN = False
USE_WANDB     = False
WANDB_PROJECT = "info8003-rl-pyflyt"
WANDB_ENTITY  = None

# ---- Naming helpers (mode is part of the run name to avoid checkpoint clashes)
def run_name(algo: str, env_short: str, mode: int, seed: int) -> str:
    return f"{algo}_{env_short}_mode{mode}_seed{seed}"

def model_path(algo: str, env_short: str, mode: int, seed: int) -> Path:
    return MODELS_DIR / f"final_{run_name(algo, env_short, mode, seed)}"

# ---- Reminder for the grader -----------------------------------------------
# scripts/evaluate.py defaults to --flight_mode 0; run it with the matching
# --flight_mode {MODES[i]} for each checkpoint, otherwise the policy will be
# evaluated under a different MDP than it was trained on.


## 3. Run training across all (seed, mode) pairs

Training is delegated to **`training_cell_v2.py`**, a self-contained module
that writes a standalone `_train_worker.py` script to disk and runs it once
per `(seed, mode)` via `subprocess.run([sys.executable, ...])`. Each
subprocess is a fresh Python interpreter, so PyBullet socket leaks die with
the process and the parent stays clean. This avoids the
`multiprocessing` + `spawn` failure mode where the worker cannot reimport
functions from the IPython kernel's `__main__`.

**Before running this cell, save `training_cell_v2.py` next to this notebook**
(or anywhere on `sys.path`).

We iterate **mode-major, seed-minor**: all 5 seeds of mode 0 first, then all 5
seeds of mode 6. This makes it possible to start analysing partial results
(once mode 0 is done) without waiting for mode 6.

In [ ]:
from training_cell_v2 import TrainingConfig, setup_training_script, train_all

# 1. Write the standalone _train_worker.py to PROJECT_ROOT (idempotent).
setup_training_script(PROJECT_ROOT)

# 2. Bundle every notebook-level setting the worker needs.
cfg = TrainingConfig(
    algo_name=ALGO_NAME, env_short=ENV_SHORT,
    total_timesteps=TOTAL_TIMESTEPS, n_envs=N_ENVS,
    eval_freq=EVAL_FREQ, n_eval_episodes=N_EVAL_EPISODES,
    max_waypoints=MAX_WAYPOINTS,
    ppo_kwargs=PPO_KWARGS, env_kwargs=WAYPOINT_KWARGS,
    scripts_dir=SCRIPTS_DIR, models_dir=MODELS_DIR, logs_dir=LOGS_DIR,
    project_root=PROJECT_ROOT,
    force_retrain=FORCE_RETRAIN,
    timeout_per_seed=20 * 60,            # 20 min — generous for 1M timesteps
)

# 3. Run all (mode, seed) pairs. Returns:
#       checkpoint_paths : dict[int -> list[Path]]   (one list per mode)
#       failed           : list[tuple[int, int]]     (mode, seed) pairs that failed
# Re-running this cell skips any (mode, seed) whose checkpoint already exists.
checkpoint_paths, failed = train_all(MODES, SEEDS, cfg)


## 4. Aggregate per-(mode, seed) evaluation logs

`EvalCallback` writes one `evaluations.npz` per (mode, seed). We collapse
across episodes (mean per evaluation point) and stack across seeds, giving
a `(n_seeds, n_evals)` matrix per mode.

In [ ]:
def load_eval_curves_for_mode(mode: int, seeds=SEEDS):
    """Return (timesteps, returns) where returns has shape (n_seeds, n_evals)."""
    all_returns, ref_ts = [], None
    for s in seeds:
        path = LOGS_DIR / run_name(ALGO_NAME, ENV_SHORT, mode, s) / "evaluations.npz"
        if not path.exists():
            print(f"[warn] missing eval log: mode {mode} seed {s}")
            continue
        data = np.load(path)
        ts, results = data["timesteps"], data["results"]
        per_eval_mean = results.mean(axis=1)
        # Defensive truncation if seeds disagree on n_evals (shouldn't happen):
        if ref_ts is None:
            ref_ts = ts
        elif len(ts) != len(ref_ts):
            n = min(len(ts), len(ref_ts))
            ref_ts = ref_ts[:n]
            all_returns = [r[:n] for r in all_returns]
            per_eval_mean = per_eval_mean[:n]
        all_returns.append(per_eval_mean)
    if not all_returns:
        raise RuntimeError(f"No eval logs found for mode {mode}.")
    return ref_ts, np.stack(all_returns, axis=0)


curves = {m: load_eval_curves_for_mode(m) for m in MODES}
for m in MODES:
    ts, returns = curves[m]
    print(f"mode {m}: timesteps={ts.shape}, returns={returns.shape}, "
          f"final-eval mean across seeds = {returns[:, -1].mean():.2f}"
          f" ± {returns[:, -1].std():.2f}")


## 5. Per-mode learning curves with bootstrap CIs

One figure per mode — same style as Task 1 (mean across seeds, shaded 95 % bootstrap CI,
thin per-seed lines). The two figures share the same y-axis range for easy visual comparison.

In [ ]:
import matplotlib.pyplot as plt


def bootstrap_ci(x: np.ndarray, n_boot: int = 5000, ci: float = 95.0,
                 stat=np.mean, rng: np.random.Generator | None = None):
    """Percentile-bootstrap CI for `stat(x)` over axis 0."""
    if rng is None:
        rng = np.random.default_rng(0)
    n = x.shape[0]
    boot = np.empty(n_boot)
    for i in range(n_boot):
        boot[i] = stat(x[rng.integers(0, n, size=n)], axis=0)
    alpha = (100.0 - ci) / 2.0
    return np.percentile(boot, alpha), np.percentile(boot, 100 - alpha)


def curve_with_ci(returns: np.ndarray, n_boot: int = 2000):
    rng = np.random.default_rng(0)
    n_seeds, n_evals = returns.shape
    mean = returns.mean(axis=0)
    low  = np.empty(n_evals); high = np.empty(n_evals)
    for j in range(n_evals):
        low[j], high[j] = bootstrap_ci(returns[:, j], n_boot=n_boot, rng=rng)
    return mean, low, high


# Compute a shared y-axis spanning both modes so they are visually comparable.
all_finals = np.concatenate([curves[m][1].flatten() for m in MODES])
y_lo, y_hi = all_finals.min() - 50, all_finals.max() + 50

for mode in MODES:
    ts, returns = curves[mode]
    mean, lo, hi = curve_with_ci(returns)

    fig, ax = plt.subplots(figsize=(8, 5))
    for i, s in enumerate(SEEDS):
        if i < len(returns):
            ax.plot(ts, returns[i], alpha=0.25, lw=1)
    ax.plot(ts, mean, lw=2.2, color="C0", label="mean across seeds")
    ax.fill_between(ts, lo, hi, alpha=0.20, color="C0", label="95 % bootstrap CI")
    ax.set_xlabel("Environment steps")
    ax.set_ylabel("Mean evaluation return")
    ax.set_ylim(y_lo, y_hi)
    ax.set_title(f"{ALGO_NAME} on {ENV_SHORT} (flight mode {mode}, {len(SEEDS)} seeds)")
    ax.grid(alpha=0.3)
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig_path = FIGURES_DIR / f"learning_curve_{ALGO_NAME}_Waypoints_mode{mode}.png"
    fig.savefig(fig_path, dpi=150)
    print(f"Saved {fig_path}")
    plt.show()


## 6. Cross-mode comparison (the headline figure for Section 5 of the report)

This is the figure that directly answers the project's *"why is mode 0 harder
than mode 6"* question. Both modes are overlaid on the same axes with the
same compute budget, so the gap between the two means is the gap due to the
control abstraction alone.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
mode_colors = {0: "C3", 6: "C2"}     # mode 0 red-ish, mode 6 green-ish
for mode in MODES:
    ts, returns = curves[mode]
    mean, lo, hi = curve_with_ci(returns)
    ax.plot(ts, mean, lw=2.2, color=mode_colors[mode], label=f"mode {mode}")
    ax.fill_between(ts, lo, hi, alpha=0.18, color=mode_colors[mode])
ax.set_xlabel("Environment steps")
ax.set_ylabel("Mean evaluation return")
ax.set_title(f"{ALGO_NAME} on {ENV_SHORT}: flight-mode comparison ({len(SEEDS)} seeds each)")
ax.grid(alpha=0.3)
ax.legend(loc="lower right", title="flight mode")
fig.tight_layout()
fig_path = FIGURES_DIR / f"learning_curve_compare_modes.png"
fig.savefig(fig_path, dpi=150)
print(f"Saved {fig_path}")
plt.show()


## 7. Final evaluation (matches `scripts/evaluate.py`)

We re-evaluate each final checkpoint over 20 deterministic episodes with
environment seeds 100..119, mirroring `scripts/evaluate.py`. For Waypoints
we additionally record `info["num_targets_reached"]` per episode — this is
arguably more interpretable than raw return for this task.

In [ ]:
def evaluate_checkpoint(ckpt_no_ext: Path, flight_mode: int,
                        n_episodes: int = FINAL_EVAL_EPISODES) -> dict:
    """Re-evaluate a single checkpoint. Reports return, length, crash, waypoints."""
    model = PPO.load(str(ckpt_no_ext))
    env = gymnasium.make(ENV_ID, flight_mode=flight_mode, **WAYPOINT_KWARGS)
    env = FlattenWaypointEnv(env, max_waypoints=MAX_WAYPOINTS)

    ep_returns, ep_lengths, ep_crashed, ep_waypoints = [], [], [], []
    for i in range(n_episodes):
        obs, info = env.reset(seed=100 + i)
        total, steps, crashed = 0.0, 0, False
        last_info = info
        while True:
            action, _ = model.predict(obs, deterministic=True)
            obs, r, terminated, truncated, info = env.step(action)
            total += r; steps += 1
            last_info = info
            if terminated:
                crashed = r <= -50           # same heuristic as scripts/evaluate.py
                break
            if truncated:
                break
        ep_returns.append(total); ep_lengths.append(steps); ep_crashed.append(crashed)
        ep_waypoints.append(int(last_info.get("num_targets_reached", 0)))
    env.close()
    return {
        "checkpoint": str(ckpt_no_ext),
        "flight_mode": flight_mode,
        "n_episodes": n_episodes,
        "ep_returns": [float(x) for x in ep_returns],
        "ep_waypoints": ep_waypoints,
        "mean_return":   float(np.mean(ep_returns)),
        "std_return":    float(np.std(ep_returns)),
        "median_return": float(np.median(ep_returns)),
        "mean_length":   float(np.mean(ep_lengths)),
        "crash_rate":    float(np.mean(ep_crashed)),
        "mean_waypoints": float(np.mean(ep_waypoints)),
        "max_waypoints":  MAX_WAYPOINTS,
    }


final_results: dict[int, dict[int, dict]] = {m: {} for m in MODES}
for mode in MODES:
    print(f"\n--- mode {mode} ---")
    for seed in SEEDS:
        ckpt = model_path(ALGO_NAME, ENV_SHORT, mode, seed)
        if not ckpt.with_suffix(".zip").exists():
            print(f"  [missing] seed {seed}")
            continue
        print(f"  [eval] seed {seed}")
        res = evaluate_checkpoint(ckpt, flight_mode=mode)
        final_results[mode][seed] = res
        out = EVAL_DIR / f"{run_name(ALGO_NAME, ENV_SHORT, mode, seed)}.json"
        with open(out, "w") as f:
            json.dump(res, f, indent=2)
        print(f"    return={res['mean_return']:.1f}  "
              f"waypoints={res['mean_waypoints']:.2f}/{MAX_WAYPOINTS}  "
              f"crash={res['crash_rate']*100:.0f}%  "
              f"len={res['mean_length']:.0f}")


## 8. Aggregate statistics per mode

Mean / IQM / bootstrap 95 % CI across seeds, computed both on episode
returns and on the number of waypoints reached. The per-mode `summary_*.json`
files are the numbers to drop into Section 4.3 of the report.

In [ ]:
def iqm(x: np.ndarray) -> float:
    q1, q3 = np.percentile(x, [25, 75])
    middle = x[(x >= q1) & (x <= q3)]
    return float(middle.mean()) if len(middle) else float(x.mean())


def aggregate(values: np.ndarray) -> dict:
    lo, hi = bootstrap_ci(values)
    return {
        "mean": float(values.mean()), "std": float(values.std()),
        "iqm": iqm(values),
        "ci95_low": float(lo), "ci95_high": float(hi),
    }


summaries = {}
for mode in MODES:
    seeds_done = sorted(final_results[mode].keys())
    if not seeds_done:
        print(f"[warn] no results for mode {mode}, skipping aggregate.")
        continue
    per_seed_returns   = np.array([final_results[mode][s]["mean_return"]    for s in seeds_done])
    per_seed_waypoints = np.array([final_results[mode][s]["mean_waypoints"] for s in seeds_done])
    per_seed_crash     = np.array([final_results[mode][s]["crash_rate"]     for s in seeds_done])
    all_eps_returns    = np.concatenate([final_results[mode][s]["ep_returns"]   for s in seeds_done])
    all_eps_waypoints  = np.concatenate([final_results[mode][s]["ep_waypoints"] for s in seeds_done])

    summary = {
        "algo": ALGO_NAME, "env": ENV_SHORT, "flight_mode": mode,
        "seeds": seeds_done, "n_episodes_total": int(len(all_eps_returns)),
        "per_seed_means_return":    per_seed_returns.tolist(),
        "per_seed_means_waypoints": per_seed_waypoints.tolist(),
        "across_seed_return":     aggregate(per_seed_returns),
        "across_seed_waypoints":  aggregate(per_seed_waypoints),
        "across_episode_return":  aggregate(all_eps_returns),
        "across_episode_waypoints": aggregate(all_eps_waypoints.astype(float)),
        "crash_rate_mean": float(per_seed_crash.mean()),
    }
    summaries[mode] = summary
    out = EVAL_DIR / f"summary_{ALGO_NAME}_{ENV_SHORT}_mode{mode}.json"
    with open(out, "w") as f:
        json.dump(summary, f, indent=2)

    print(f"\n=== mode {mode} ===")
    print(f"Per-seed mean returns:   {per_seed_returns.round(1).tolist()}")
    print(f"Per-seed mean waypoints: {per_seed_waypoints.round(2).tolist()}")
    print(f"Across seeds — return:    "
          f"mean={summary['across_seed_return']['mean']:.1f}  "
          f"IQM={summary['across_seed_return']['iqm']:.1f}  "
          f"95%CI=[{summary['across_seed_return']['ci95_low']:.1f}, "
          f"{summary['across_seed_return']['ci95_high']:.1f}]")
    print(f"Across seeds — waypoints: "
          f"mean={summary['across_seed_waypoints']['mean']:.2f}/{MAX_WAYPOINTS}  "
          f"IQM={summary['across_seed_waypoints']['iqm']:.2f}  "
          f"95%CI=[{summary['across_seed_waypoints']['ci95_low']:.2f}, "
          f"{summary['across_seed_waypoints']['ci95_high']:.2f}]")
    print(f"Crash rate: {summary['crash_rate_mean']*100:.1f}%")


## 9. Per-mode and cross-mode distributions

Two figures:

* **Per-mode box-plots** of episode return, one per mode (same as Task 1's
  final-eval boxplot).
* **Cross-mode comparison of waypoints reached** — discrete y-axis from 0 to
  `MAX_WAYPOINTS=4`, makes the difficulty gap visceral.

In [ ]:
# (a) Per-mode return boxplots
for mode in MODES:
    if not final_results[mode]:
        continue
    seeds_done = sorted(final_results[mode].keys())
    fig, ax = plt.subplots(figsize=(7, 4.5))
    data = [final_results[mode][s]["ep_returns"] for s in seeds_done]
    ax.boxplot(data, labels=[f"seed {s}" for s in seeds_done], showmeans=True)
    ax.set_ylabel("Episode return")
    ax.set_title(f"{ALGO_NAME} on {ENV_SHORT} (mode {mode}) — final eval "
                 f"({FINAL_EVAL_EPISODES} eps/seed)")
    ax.grid(alpha=0.3, axis="y")
    fig.tight_layout()
    fig_path = FIGURES_DIR / f"final_eval_box_return_mode{mode}.png"
    fig.savefig(fig_path, dpi=150)
    print(f"Saved {fig_path}")
    plt.show()

# (b) Cross-mode waypoints-reached comparison
fig, ax = plt.subplots(figsize=(7.5, 5))
positions, labels, all_data = [], [], []
xpos = 0
for mode in MODES:
    if not final_results[mode]:
        continue
    seeds_done = sorted(final_results[mode].keys())
    for s in seeds_done:
        all_data.append(final_results[mode][s]["ep_waypoints"])
        positions.append(xpos); labels.append(f"m{mode}\ns{s}"); xpos += 1
    xpos += 1   # gap between modes
ax.boxplot(all_data, positions=positions, labels=labels, showmeans=True)
ax.set_ylabel("Waypoints reached (out of 4)")
ax.set_ylim(-0.3, MAX_WAYPOINTS + 0.3)
ax.set_yticks(range(MAX_WAYPOINTS + 1))
ax.set_title(f"{ALGO_NAME} on {ENV_SHORT}: waypoints reached, by (mode, seed)")
ax.grid(alpha=0.3, axis="y")
ax.axhline(MAX_WAYPOINTS, color="gray", lw=0.8, ls="--", alpha=0.5)
fig.tight_layout()
fig_path = FIGURES_DIR / f"waypoints_reached_compare_modes.png"
fig.savefig(fig_path, dpi=150)
print(f"Saved {fig_path}")
plt.show()


## 10. Optional — render a deterministic rollout

Renders one episode from the best (mode, seed) combination. Requires a
display; off by default.

In [ ]:
DO_RENDER = False

if DO_RENDER:
    # Pick the best (mode, seed) by mean waypoints reached, breaking ties on return.
    best = None; best_key = (-1.0, -1.0)
    for mode in MODES:
        for s, res in final_results[mode].items():
            key = (res["mean_waypoints"], res["mean_return"])
            if key > best_key:
                best_key = key
                best = (mode, s)
    mode, seed = best
    print(f"Rendering best: mode {mode} seed {seed}  "
          f"(waypoints={best_key[0]:.2f}, return={best_key[1]:.1f})")

    model = PPO.load(str(model_path(ALGO_NAME, ENV_SHORT, mode, seed)))
    env = gymnasium.make(ENV_ID, flight_mode=mode, render_mode="human", **WAYPOINT_KWARGS)
    env = FlattenWaypointEnv(env, max_waypoints=MAX_WAYPOINTS)
    obs, _ = env.reset(seed=42)
    total = 0.0; steps = 0; final_info = {}
    while True:
        action, _ = model.predict(obs, deterministic=True)
        obs, r, terminated, truncated, info = env.step(action)
        total += r; steps += 1; final_info = info
        if terminated or truncated:
            break
    env.close()
    print(f"return={total:.1f}, length={steps}, "
          f"waypoints={final_info.get('num_targets_reached', 0)}/{MAX_WAYPOINTS}")
else:
    print("Rendering skipped (DO_RENDER=False).")


## 11. Outputs

After running this notebook end-to-end you will have, **per mode**:

* `results/models/final_PPO_QuadX-Waypoints-v4_mode{0,6}_seed{0..4}.zip` — final checkpoints
  (loadable by `scripts/evaluate.py --flight_mode <mode>`).
* `results/logs/PPO_QuadX-Waypoints-v4_mode{0,6}_seed{0..4}/evaluations.npz` — eval curves.
* `results/eval/PPO_QuadX-Waypoints-v4_mode{0,6}_seed{0..4}.json` — per-seed final-eval stats.
* `results/eval/summary_PPO_QuadX-Waypoints-v4_mode{0,6}.json` — aggregate statistics.
* `results/figures/learning_curve_PPO_Waypoints_mode{0,6}.png` — per-mode learning curves.
* `results/figures/learning_curve_compare_modes.png` — **headline cross-mode figure**.
* `results/figures/final_eval_box_return_mode{0,6}.png` — per-mode return box-plots.
* `results/figures/waypoints_reached_compare_modes.png` — discrete waypoints-reached comparison.

These feed into Section 4.3 (per-mode results) and Section 5 (cross-mode analysis) of the report
of the report.